# 生成天气图表 （确认好时间范围）

In [7]:
# 依赖包
from src import meteo_config_im as config
from src import weather_charts_builder

# 基础数据
countries = config.countries  # 涉及的国家、地区
daily_indicators = config.daily_indicators  #需要查询的指标

# 存档的历史数据
id_path = ['dataset/ID_2011-2020.csv',
           'dataset/ID_2021-2024.csv',
           'dataset/ID_202501-09.csv']
my_path = ['dataset/MY_2011-2020.csv',
           'dataset/MY_2021-2024.csv',
           'dataset/MY_202501-09.csv']
countries[0]['csv_path'] = id_path
countries[1]['csv_path'] = my_path

# 绘制保存图表
params = {
    'countries': countries,
    'csv_start_date': '2015-01-01',
    'api_start_date': '2025-10-01',
    'api_end_date': '2025-10-11',
    'forecast_start_date': '2025-10-12',
    'forecast_end_date': '2025-10-26',
    'daily_indicators': daily_indicators,
    'config': config
}
weather_charts_builder.with_forecast(**params)

.. Loading: Riau
.. Loading: North Sumatra
.. Loading: Central Kalimantan
.. Loading: East Kalimantan
.. Loading: West Kalimantan
.. Loading: Jambi
.. Loading: South Sumatra
Finished: Indonesia
.. Loading: Sabah
.. Loading: Sarawak
.. Loading: Johor
.. Loading: Pahang
.. Loading: Parak
Finished: Malaysia


# 合成大图

In [8]:
from src import meteo_config_im as config
from src import charts

for country in config.countries:
    file_lists = []
    for city in country['city_list']:
        for style in config.styles:
            file_lists.append(f'diagram/{country['code']}/{country['code']}{city['code']}_{style['path']}.jpg')
    charts.merge2grid(file_lists, len(country['city_list']), len(config.styles), f'diagram/grid/{country['code']}.jpg')

print('合并完成...')

合并完成...


# 周报用图

In [9]:
from src import meteo_config_im as config
from src import charts
s1 = [config.styles[0]['path'], #累计降水
      config.styles[1]['path'], #7日降水
      config.styles[3]['path'], #墒情
      config.styles[4]['path'], #温度
      config.styles[2]['path']] #30日降水
s2 = [config.styles[1]['path'], #7日降水
      config.styles[3]['path'],]#墒情
# 天气周报
file_lists = []
for country in countries:
    for city in country['city_list']:
        for s in s1:
            file_lists.append(f'diagram/{country['code']}/{country['code']}{city['code']}_{s}.jpg')
charts.merge2grid(file_lists[ 0:15], 3, len(s1), f'diagram/grid/wr1.jpg')
charts.merge2grid(file_lists[15:30], 3, len(s1), f'diagram/grid/wr2.jpg')
charts.merge2grid(file_lists[30:45], 3, len(s1), f'diagram/grid/wr3.jpg')
charts.merge2grid(file_lists[45:60], 3, len(s1), f'diagram/grid/wr4.jpg')
# 天气简报
file_lists = []
for contry in countries:
    for city in contry['city_list']:
        for s in s2:
            file_lists.append(f'diagram/{contry['code']}/{contry['code']}{city['code']}_{s}.jpg')
charts.merge2grid(file_lists[ 0:14], 7, len(s2), f'diagram/grid/wr5.jpg')
charts.merge2grid(file_lists[14:24], 5, len(s2), f'diagram/grid/wr6.jpg')

print('合并完成...')

合并完成...


# 保存历史数据

In [ ]:
import pandas as pd
from src import om_api
from src import meteo_config_im as config

# 确定需要存档的「时间范围」与「文件后缀」即可运行
start_date = '2025-09-01'
end_date = '2025-09-30'
tial_of_file = '202509'

for country in config.countries:
    all_df = pd.DataFrame()
    for city  in country['city_list']:
        params = {
            'latitude'  : city['latitude'],
            'longitude' : city['longitude'],
            'start_date': start_date,
            'end_date'  : end_date,
            'daily_indicators': config.daily_indicators
        }
        api_df = om_api.daily_history(**params)
        groupby_df = api_df.groupby('date', as_index=False)[config.daily_indicators].mean()
        groupby_df['name'] = city['name']
        print('.. Loading: ' + city['name'])
        all_df = pd.concat([all_df, groupby_df])
    all_df.to_csv(f'dataset/{country['code']}_{tial_of_file}.csv', index=False)
    print('Finished: ' + country['name'])

# 合并历史数据文件

In [ ]:
import pandas as pd
from src import meteo_config_im as config

tail_of_input_file = ['202501-202508','202509']
tail_of_output_file = '202501-09'

for country in config.countries:
    out_df = pd.DataFrame()
    for t in tail_of_input_file:
        in_df = pd.read_csv(f'dataset/{country['code']}_{t}.csv', sep = ',')
        out_df = pd.concat([out_df, in_df])
    out_df.to_csv(f'dataset/{country["code"]}_{tail_of_output_file}.csv', index=False)